# Serie III - Ejercicio 3.1: PySpark con Adventure Works

**Dataset:** [Adventure Works in Excel Tables](https://www.kaggle.com/datasets/algorismus/adventure-works-in-excel-tables)

**Restriccion:** NO usar comandos ni funciones de Spark SQL. Toda manipulacion debe ser con PySpark DataFrames.

**Tareas:**
1. Ventas por region comparado anualmente
2. Top 2 productos mas vendidos por año
3. Empleado con mayor ventas por mes (2020)

## Configuracion del Entorno

In [53]:
# Instalacion de dependencias
!pip install pyspark pandas

In [54]:
# Importaciones necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, sum as spark_sum, count,
    row_number, dense_rank, first, concat, lit,
    when, coalesce, round as spark_round,
    format_number, to_date, regexp_replace, trim,
    split, element_at
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType
)
import os

In [55]:
# Inicializar SparkSession
spark = SparkSession.builder \
    .appName("AdventureWorks_Analysis") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("SparkSession creada exitosamente.")

Spark Version: 4.0.1
SparkSession creada exitosamente.


## Carga de Datos

El dataset de Adventure Works contiene archivos CSV:
- `Product.csv`
- `Region.csv`
- `Reseller.csv`
- `Sales.csv`
- `Salesperson.csv`
- `SalespersonRegion.csv`
- `Targets.csv`



In [56]:
# Ruta base de los archivos
# Para local:
# DATA_PATH = 'data/'

# Para Google Colab, subir los archivos y usar:
DATA_PATH = '/content/data/'

In [57]:
# Cargar los DataFrames (archivos TSV - Tab Separated Values)
print("Cargando datos de Adventure Works...")
print("=" * 50)

# Cargar ventas
sales_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .csv(f"{DATA_PATH}Sales.csv")
print(f"Sales: {sales_df.count()} registros")

# Cargar productos
products_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .csv(f"{DATA_PATH}Product.csv")
print(f"Products: {products_df.count()} registros")

# Cargar regiones
regions_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .csv(f"{DATA_PATH}Region.csv")
print(f"Regions: {regions_df.count()} registros")

# Cargar vendedores
salesperson_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "true") \
    .csv(f"{DATA_PATH}Salesperson.csv")
print(f"Salesperson: {salesperson_df.count()} registros")

print("=" * 50)
print("Datos cargados exitosamente!")

Cargando datos de Adventure Works...
Sales: 57851 registros
Products: 397 registros
Regions: 10 registros
Salesperson: 18 registros
Datos cargados exitosamente!


In [58]:
# Mostrar esquemas de los DataFrames
print("\n--- Esquema de Sales ---")
sales_df.printSchema()

print("\n--- Esquema de Products ---")
products_df.printSchema()

print("\n--- Esquema de Regions ---")
regions_df.printSchema()

print("\n--- Esquema de Salesperson ---")
salesperson_df.printSchema()


--- Esquema de Sales ---
root
 |-- SalesOrderNumber: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- ProductKey: integer (nullable = true)
 |-- ResellerKey: integer (nullable = true)
 |-- EmployeeKey: integer (nullable = true)
 |-- SalesTerritoryKey: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit Price: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Cost: string (nullable = true)


--- Esquema de Products ---
root
 |-- ProductKey: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Standard Cost: string (nullable = true)
 |-- Color: string (nullable = true)
 |-- Subcategory: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Background Color Format: string (nullable = true)
 |-- Font Color Format: string (nullable = true)


--- Esquema de Regions ---
root
 |-- SalesTerritoryKey: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = tr

In [59]:
# Vista previa de los datos
print("\n--- Vista Previa: Sales ---")
sales_df.show(5, truncate=False)

print("\n--- Vista Previa: Products ---")
products_df.show(5, truncate=False)

print("\n--- Vista Previa: Regions ---")
regions_df.show(5, truncate=False)

print("\n--- Vista Previa: Salesperson ---")
salesperson_df.show(5, truncate=False)


--- Vista Previa: Sales ---
+----------------+---------------------------+----------+-----------+-----------+-----------------+--------+----------+---------+---------+
|SalesOrderNumber|OrderDate                  |ProductKey|ResellerKey|EmployeeKey|SalesTerritoryKey|Quantity|Unit Price|Sales    |Cost     |
+----------------+---------------------------+----------+-----------+-----------+-----------------+--------+----------+---------+---------+
|SO43897         |Friday, August 25, 2017    |235       |312        |282        |4                |2       |$28.84    |$57.68   |$63.45   |
|SO43897         |Friday, August 25, 2017    |351       |312        |282        |4                |2       |$2,024.99 |$4,049.98|$3,796.19|
|SO43897         |Friday, August 25, 2017    |348       |312        |282        |4                |2       |$2,024.99 |$4,049.98|$3,796.19|
|SO43897         |Friday, August 25, 2017    |232       |312        |282        |4                |2       |$28.84    |$57.68   |$6

## Preparacion de Datos

Transformamos los datos para facilitar el analisis:
- Parsear la fecha de OrderDate
- Limpiar valores monetarios
- Extraer año y mes

In [60]:
# Parsear la fecha (formato: "Friday, August 25, 2017")
# Extraer solo la parte de fecha sin el dia de la semana
# Al hacer split por ", " obtenemos: ["Friday", "August 25", "2017"]
# Necesitamos concatenar indices 1 y 2 para obtener "August 25, 2017"

sales_clean = sales_df \
    .withColumn(
        'date_part',
        concat(
            element_at(split(col('OrderDate'), ', '), 2),  # "August 25"
            lit(', '),
            element_at(split(col('OrderDate'), ', '), 3)   # "2017"
        )
    ) \
    .withColumn(
        'OrderDateClean',
        to_date(col('date_part'), 'MMMM d, yyyy')
    ) \
    .withColumn('año', year(col('OrderDateClean'))) \
    .withColumn('mes', month(col('OrderDateClean'))) \
    .withColumn(
        'SalesAmount',
        regexp_replace(col('Sales'), '[\\$,]', '').cast('double')
    ) \
    .drop('date_part')

# Verificar transformacion
print("Datos de ventas transformados:")
sales_clean.select('OrderDate', 'OrderDateClean', 'año', 'mes', 'Sales', 'SalesAmount').show(5, truncate=False)

Datos de ventas transformados:
+---------------------------+--------------+----+---+---------+-----------+
|OrderDate                  |OrderDateClean|año |mes|Sales    |SalesAmount|
+---------------------------+--------------+----+---+---------+-----------+
|Friday, August 25, 2017    |2017-08-25    |2017|8  |$57.68   |57.68      |
|Friday, August 25, 2017    |2017-08-25    |2017|8  |$4,049.98|4049.98    |
|Friday, August 25, 2017    |2017-08-25    |2017|8  |$4,049.98|4049.98    |
|Friday, August 25, 2017    |2017-08-25    |2017|8  |$57.68   |57.68      |
|Saturday, November 18, 2017|2017-11-18    |2017|11 |$1,637.4 |1637.4     |
+---------------------------+--------------+----+---+---------+-----------+
only showing top 5 rows


In [61]:
# Verificar años disponibles
print("Años disponibles en los datos:")
sales_clean.select('año').distinct().orderBy('año').show()

Años disponibles en los datos:
+----+
| año|
+----+
|2017|
|2018|
|2019|
|2020|
+----+



---
## Tarea 1: Ventas por Region Comparado Anualmente

**Formato esperado:**
```
| region | Año_2017 | Año_2018 | Año_2019 | Año_2020 |
```

In [62]:
print("=" * 70)
print("TAREA 1: Ventas por Region Comparado Anualmente")
print("=" * 70)

# Unir ventas con regiones
ventas_region = sales_clean.join(
    regions_df,
    sales_clean['SalesTerritoryKey'] == regions_df['SalesTerritoryKey'],
    'inner'
)

# Agrupar por region y año, sumar ventas
ventas_por_region_año = ventas_region.groupBy('Region', 'año') \
    .agg(spark_sum('SalesAmount').alias('total_ventas'))

# Obtener años unicos
años_disponibles = [row['año'] for row in ventas_por_region_año.select('año').distinct().collect()]
años_disponibles.sort()
print(f"\nAños disponibles: {años_disponibles}")

pivot_df = ventas_por_region_año.groupBy('Region') \
    .agg(
        spark_round(spark_sum(when(col('año') == 2017, col('total_ventas')).otherwise(0)), 2).alias('Año_2017'),
        spark_round(spark_sum(when(col('año') == 2018, col('total_ventas')).otherwise(0)), 2).alias('Año_2018'),
        spark_round(spark_sum(when(col('año') == 2019, col('total_ventas')).otherwise(0)), 2).alias('Año_2019'),
        spark_round(spark_sum(when(col('año') == 2020, col('total_ventas')).otherwise(0)), 2).alias('Año_2020')
    ) \
    .orderBy('Region')

print("\n--- Ventas por Region (Comparacion Anual) ---")
pivot_df.show(truncate=False)

TAREA 1: Ventas por Region Comparado Anualmente

Años disponibles: [2017, 2018, 2019, 2020]

--- Ventas por Region (Comparacion Anual) ---
+--------------+----------+----------+----------+----------+
|Region        |Año_2017  |Año_2018  |Año_2019  |Año_2020  |
+--------------+----------+----------+----------+----------+
|Australia     |0.0       |0.0       |875621.82 |515403.03 |
|Canada        |1514285.89|4862018.61|5691150.09|1808178.37|
|Central       |951799.96 |2635728.33|3016263.54|1029595.03|
|France        |0.0       |860904.52 |2406134.06|1260801.11|
|Germany       |0.0       |0.0       |1127594.53|750148.86 |
|Northeast     |568552.28 |2453772.16|2876166.6 |816863.35 |
|Northwest     |1690237.48|3502591.03|4687348.5 |2124645.44|
|Southeast     |1450813.3 |2838535.04|2444552.75|904705.46 |
|Southwest     |1894066.67|6330990.61|7188401.7 |2587657.12|
|United Kingdom|0.0       |844245.95 |2186771.99|852025.02 |
+--------------+----------+----------+----------+----------+



---
## Tarea 2: Top 2 Productos Mas Vendidos por Año

**Formato esperado:**
```
| Nombre_producto | año | unidades_vendidas |
```

In [63]:
print("\n" + "=" * 70)
print("TAREA 2: Top 2 Productos Mas Vendidos por Año")
print("=" * 70)

# Unir ventas con productos
ventas_productos = sales_clean.join(
    products_df,
    sales_clean['ProductKey'] == products_df['ProductKey'],
    'inner'
)

# Agrupar por producto y año, sumar unidades vendidas
ventas_por_producto_año = ventas_productos.groupBy('Product', 'año') \
    .agg(spark_sum('Quantity').alias('unidades_vendidas'))

# Crear ventana para ranking por año
window_spec = Window.partitionBy('año').orderBy(col('unidades_vendidas').desc())

# Agregar ranking y filtrar top 2
top_productos = ventas_por_producto_año \
    .withColumn('ranking', row_number().over(window_spec)) \
    .filter(col('ranking') <= 2) \
    .select(
        col('Product').alias('Nombre_producto'),
        col('año'),
        col('unidades_vendidas'),
        col('ranking')
    ) \
    .orderBy('año', 'ranking')

print("\n--- Top 2 Productos Mas Vendidos por Año ---")
top_productos.show(20, truncate=False)


TAREA 2: Top 2 Productos Mas Vendidos por Año

--- Top 2 Productos Mas Vendidos por Año ---
+-------------------------------+----+-----------------+-------+
|Nombre_producto                |año |unidades_vendidas|ranking|
+-------------------------------+----+-----------------+-------+
|Mountain Bike Socks, M         |2017|563              |1      |
|AWC Logo Cap                   |2017|520              |2      |
|Full-Finger Gloves, L          |2018|1942             |1      |
|Long-Sleeve Logo Jersey, L     |2018|1908             |2      |
|AWC Logo Cap                   |2019|2677             |1      |
|Long-Sleeve Logo Jersey, L     |2019|2658             |2      |
|Classic Vest, S                |2020|1349             |1      |
|Short-Sleeve Classic Jersey, XL|2020|1107             |2      |
+-------------------------------+----+-----------------+-------+



---
## Tarea 3: Empleado con Mayor Ventas por Mes (2020)

**Formato esperado:**
```
| Id_empleado | Nombre_empleado | mes | total_ventas |
```


In [73]:
print("\n" + "=" * 70)
print("TAREA 3: Empleado con Mayor Ventas por Mes")
print("=" * 70)

# Verificar datos por año
print("\nRegistros por año:")
sales_clean.groupBy('año').count().orderBy('año').show()

año_analisis = 2020
registros_2020 = sales_clean.filter(col('año') == 2020).count()

print(f"\nUsando año {año_analisis} para el analisis.")


TAREA 3: Empleado con Mayor Ventas por Mes

Registros por año:
+----+-----+
| año|count|
+----+-----+
|2017| 4138|
|2018|16676|
|2019|26758|
|2020|10279|
+----+-----+


Usando año 2020 para el analisis.


In [67]:
# Filtrar ventas del año seleccionado
ventas_año = sales_clean.filter(col('año') == año_analisis)

# Agrupar ventas por empleado y mes
ventas_empleado_mes = ventas_año.groupBy('EmployeeKey', 'mes') \
    .agg(spark_round(spark_sum('SalesAmount'), 2).alias('total_ventas'))

# Crear ventana para ranking por mes
window_mes = Window.partitionBy('mes').orderBy(col('total_ventas').desc())

# Obtener el top 1 por mes
top_empleado_mes = ventas_empleado_mes \
    .withColumn('ranking', row_number().over(window_mes)) \
    .filter(col('ranking') == 1)

# Unir con informacion de empleados
resultado_empleados = top_empleado_mes.join(
    salesperson_df,
    top_empleado_mes['EmployeeKey'] == salesperson_df['EmployeeKey'],
    'inner'
).select(
    top_empleado_mes['EmployeeKey'].alias('Id_empleado'),
    col('Salesperson').alias('Nombre_empleado'),
    col('mes'),
    col('total_ventas')
).orderBy('mes')

print(f"\n--- Empleado con Mayor Ventas por Mes ({año_analisis}) ---")
resultado_empleados.show(12, truncate=False)


--- Empleado con Mayor Ventas por Mes (2020) ---
+-----------+------------------------+---+------------+
|Id_empleado|Nombre_empleado         |mes|total_ventas|
+-----------+------------------------+---+------------+
|291        |Jae Pak                 |1  |219460.4    |
|282        |Linda Mitchell          |2  |499364.86   |
|291        |Jae Pak                 |3  |353918.0    |
|291        |Jae Pak                 |4  |370602.3    |
|292        |Ranjit Varkey Chudukatil|5  |608336.73   |
+-----------+------------------------+---+------------+



In [68]:
# Agregar nombre del mes para mejor visualizacion
resultado_con_nombre_mes = resultado_empleados.withColumn(
    'Nombre_Mes',
    when(col('mes') == 1, 'Enero')
    .when(col('mes') == 2, 'Febrero')
    .when(col('mes') == 3, 'Marzo')
    .when(col('mes') == 4, 'Abril')
    .when(col('mes') == 5, 'Mayo')
    .when(col('mes') == 6, 'Junio')
    .when(col('mes') == 7, 'Julio')
    .when(col('mes') == 8, 'Agosto')
    .when(col('mes') == 9, 'Septiembre')
    .when(col('mes') == 10, 'Octubre')
    .when(col('mes') == 11, 'Noviembre')
    .when(col('mes') == 12, 'Diciembre')
    .otherwise('Desconocido')
)

print("\n--- Resultado con Nombre de Mes ---")
resultado_con_nombre_mes.select(
    'Id_empleado', 'Nombre_empleado', 'Nombre_Mes', 'total_ventas'
).show(12, truncate=False)


--- Resultado con Nombre de Mes ---
+-----------+------------------------+----------+------------+
|Id_empleado|Nombre_empleado         |Nombre_Mes|total_ventas|
+-----------+------------------------+----------+------------+
|291        |Jae Pak                 |Enero     |219460.4    |
|282        |Linda Mitchell          |Febrero   |499364.86   |
|291        |Jae Pak                 |Marzo     |353918.0    |
|291        |Jae Pak                 |Abril     |370602.3    |
|292        |Ranjit Varkey Chudukatil|Mayo      |608336.73   |
+-----------+------------------------+----------+------------+



---
## Resumen de Resultados

In [75]:
print("\n" + "=" * 70)
print("RESUMEN FINAL DE ANALISIS")
print("=" * 70)

print("\n" + "-" * 70)
print("TAREA 1: Ventas por Region (Comparacion Anual)")
print("-" * 70)
pivot_df.show(truncate=False)

print("\n" + "-" * 70)
print("TAREA 2: Top 2 Productos Mas Vendidos por Año")
print("-" * 70)
top_productos.show(20, truncate=False)

print("\n" + "-" * 70)
print(f"TAREA 3: Empleado con Mayor Ventas por Mes ({año_analisis})")
print("-" * 70)
resultado_empleados.show(12, truncate=False)


RESUMEN FINAL DE ANALISIS

----------------------------------------------------------------------
TAREA 1: Ventas por Region (Comparacion Anual)
----------------------------------------------------------------------
+--------------+----------+----------+----------+----------+
|Region        |Año_2017  |Año_2018  |Año_2019  |Año_2020  |
+--------------+----------+----------+----------+----------+
|Australia     |0.0       |0.0       |875621.82 |515403.03 |
|Canada        |1514285.89|4862018.61|5691150.09|1808178.37|
|Central       |951799.96 |2635728.33|3016263.54|1029595.03|
|France        |0.0       |860904.52 |2406134.06|1260801.11|
|Germany       |0.0       |0.0       |1127594.53|750148.86 |
|Northeast     |568552.28 |2453772.16|2876166.6 |816863.35 |
|Northwest     |1690237.48|3502591.03|4687348.5 |2124645.44|
|Southeast     |1450813.3 |2838535.04|2444552.75|904705.46 |
|Southwest     |1894066.67|6330990.61|7188401.7 |2587657.12|
|United Kingdom|0.0       |844245.95 |2186771.99|85

In [76]:
# Cerrar SparkSession
spark.stop()
print("\nSparkSession cerrada.")
print("\n" + "=" * 70)
print("ANALISIS COMPLETADO EXITOSAMENTE")
print("=" * 70)


SparkSession cerrada.

ANALISIS COMPLETADO EXITOSAMENTE
